**Calculate Correlation Coefficient between station observations and incident pixels**

In [1]:
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

In [2]:
datasets = ['CHIRPS', 'PERSIANN', 'ERA5L', 'HAR', 'MSWEP', 'kriged'] 

#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])
df_insitu = df_insitu[(df_insitu['year'] >= 2001) & (df_insitu['year'] <= 2005)]
df_insitu = df_insitu[(df_insitu['year'] >= 2001) & (df_insitu['year'] <= 2005)]

#Get path, files and file list for datasets
def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    if dataset == 'kriged':
        dataset_path = r"..\Data\Processed\Kriging_Predictors\Kriged_rasters"
    else:
        dataset_path = fr"..\Data\Datasets\{dataset}_prec"
    dataset_tiff_files = glob.glob(os.path.join(dataset_path, "*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tiff_files))
    return dataset_path, dataset_tiff_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tiff_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_files'] = dataset_tiff_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['MSWEP_files']

In [3]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
unique_stn.to_csv(r'../Data/unique_stn.csv',index=False)   

In [4]:
def extract_concurrent_grid_data(dataset_results, stn_points, unique_stn):
    df_final = None
    
    for dataset in datasets:
        files = dataset_results[f'{dataset}_files']
        print(f"Processing {dataset} with {len(files)} files...")
        
        def process_file(file):
            date_obj = extract_date(file, dataset) 
            local_records = []
            try:
                with rasterio.open(file) as src:
                    crs = src.crs
                    if crs == "EPSG:4326":
                        sampled_values = list(src.sample(stn_points))
                        for i, value in enumerate(sampled_values):
                            local_records.append({
                                'date': date_obj,
                                'stn_id': unique_stn.iloc[i]['index_no'],
                                'prec_grid': value[0]
                                })
            except Exception as e:
                print(f'Error processing {file}: {e}')
            return local_records

        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(process_file, files)
            for result in results:
                dataset_records.extend(result)
        
        df_dataset = pd.DataFrame(dataset_records)
        
        # CHECK IF EMPTY FIRST
        if not df_dataset.empty:
            # df_dataset = df_dataset[df_dataset['prec_grid'] != -9999.0]
            # df_dataset = df_dataset.dropna().round(1)
            df_dataset.loc[df_dataset['prec_grid'] < 0, 'prec_grid'] = 0
            df_dataset = df_dataset.rename(columns={'prec_grid': f'{dataset}_prec'})
            
            # Merge logic
            if df_final is None:
                df_final = df_dataset
            else:
                df_final = pd.merge(df_final, df_dataset, on=['date', 'stn_id'], how='inner')
                print(f"Merged {dataset}. Resulting shape: {df_final.shape}")
        else:
            print(f"Warning: No data extracted for {dataset}")

    return df_final

df_grid_data = extract_concurrent_grid_data(dataset_results, stn_points, unique_stn)

Processing CHIRPS with 10957 files...
Processing PERSIANN with 10957 files...
Merged PERSIANN. Resulting shape: (720256, 4)
Processing ERA5L with 10957 files...
Merged ERA5L. Resulting shape: (720188, 5)
Processing HAR with 10957 files...
Merged HAR. Resulting shape: (720188, 6)
Processing MSWEP with 10958 files...
Merged MSWEP. Resulting shape: (720188, 7)
Processing kriged with 1825 files...
Merged kriged. Resulting shape: (124100, 8)


In [5]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid_data['date'] = pd.to_datetime(df_grid_data['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid_data, df_insitu1, on=['date','stn_id'], how='inner')

df_merged.to_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv', index=False)

**Calculating error metrices & CPI- Daily Scale**

In [6]:
#checkpoint - run from here to reduce code runtime while rerunning
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

datasets = ['ERA5L', 'HAR', 'CHIRPS', 'MSWEP', 'PERSIANN', 'kriged']
df_merged = pd.read_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv')
df_merged = df_merged.round(1)
df_merged


,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,kriged_prec,prec_insitu
0,2001-01-01,102.0,7.8,4.7,16.0,18.7,3.8,16.6,0.0
1,2001-01-01,103.0,8.3,3.9,15.4,5.7,3.9,15.3,0.0
2,2001-01-01,104.0,9.1,3.9,16.2,18.3,4.1,15.3,0.0
3,2001-01-01,105.0,9.9,0.0,3.2,1.5,1.6,3.2,0.0
4,2001-01-01,107.0,5.9,2.8,11.8,11.6,3.7,12.0,7.0
...,...,...,...,...,...,...,...,...,...
121380,2005-12-31,1407.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121381,2005-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121382,2005-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121383,2005-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    #datasets = ['ERA5L', 'HARv2', 'CHIRPS', 'MSWEP', 'PERSIANN']

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['prec_insitu'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(prec_grid_array),
            ~np.isnan(prec_insitu_array),
            prec_grid_array >=  0,
            prec_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy_daily = calculate_accuracy(df_merged)
df_accuracy_daily

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.990170,2.0,0.563405,3.158772,30.178824,1.001365,119599
1,HAR,1.221821,3.0,0.513546,3.266117,11.637083,0.966977,119599
2,CHIRPS,2.117945,5.0,0.283558,3.260391,-11.499708,0.885801,119599
3,MSWEP,2.837427,6.0,0.408536,2.964136,-89.522095,0.137352,119599
4,PERSIANN,1.232311,4.0,0.412874,2.845784,-25.930879,0.562320,119599
5,kriged,0.977603,1.0,0.564953,3.156485,30.102029,1.000811,119599


**Accuracy and CPI Calculation - Seasonal Scale**

In [8]:
#define seasons
seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
df_merged1 = df_merged.copy()
df_merged1['date'] = pd.to_datetime(df_merged1['date'])
df_merged1['year'] = df_merged1['date'].dt.year
df_merged1['month'] = df_merged1['date'].dt.month

#create effective year for seasonal analysis
df_merged1['year_seasonal'] = df_merged1['year']
df_merged1.loc[df_merged1['month'] == 12, 'year_seasonal'] = df_merged1.loc[df_merged1['month'] == 12, 'year'] + 1

df_merged1['season'] = (df_merged1['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))

#calculate seasonal rainfall sum for each dataset 
df_seasonal = df_merged1.groupby(by = ['season','year_seasonal'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu','HAR_prec', 'MSWEP_prec', 'kriged_prec']].sum().reset_index()
df_seasonal

,season,year_seasonal,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,prec_insitu,HAR_prec,MSWEP_prec,kriged_prec
0,monsoon,2001,71456.2,58281.5,94173.0,85118.1,81523.6,7513.1,93701.4
1,monsoon,2002,79517.5,55267.7,108089.3,88916.1,89455.8,8023.7,107945.0
2,monsoon,2003,80538.5,66887.4,124850.2,94734.7,91159.2,10180.6,124731.5
3,monsoon,2004,84122.8,61486.3,99176.7,81279.8,81966.6,7720.4,99001.0
4,monsoon,2005,64814.7,48230.4,103062.9,67567.4,83297.4,7344.1,102764.8
5,post_monsoon,2001,4214.4,5994.2,10749.8,7161.8,8793.4,941.8,10674.7
6,post_monsoon,2002,2196.8,2494.3,5381.5,2622.0,4188.3,423.3,5312.8
7,post_monsoon,2003,3214.0,3183.1,7840.7,3748.3,5544.8,559.0,7750.8
8,post_monsoon,2004,4755.4,4944.5,9342.2,6342.9,12326.9,685.4,9310.7
9,post_monsoon,2005,4158.5,4643.4,8111.6,4681.1,5778.9,687.7,8069.5


In [9]:
#caluclate overall seasonal cpi
df_accuracy_seasonal = (calculate_accuracy(df_seasonal))
df_accuracy_seasonal['season'] = 'all'
#print("OVERALL SEASONAL ACCURACY:")
#print(df_accuracy_seasonal)

#calculate monsoonal cpi
df_monsoon = df_seasonal[df_seasonal['season'] == 'monsoon']
df_accuracy_monsoon = calculate_accuracy(df_monsoon)
df_accuracy_monsoon['season'] = 'monsoon'
#print("MONSOON ACCURACY:")
#print(df_accuracy_monsoon)

#calculate pre-monsoon cpi
df_premonsoon = df_seasonal[df_seasonal['season'] == 'pre_monsoon']
df_accuracy_premonsoon = calculate_accuracy(df_premonsoon)
df_accuracy_premonsoon['season'] = 'premonsoon'
#print("PRE-MONSOON ACCURACY:")
#print(df_accuracy_premonsoon)

#calculate post-monsoon cpi
df_postmonsoon = df_seasonal[df_seasonal['season'] == 'post_monsoon']
df_accuracy_postmonsoon = calculate_accuracy(df_postmonsoon)
df_accuracy_postmonsoon['season'] = 'postmonsoon'
#print("POST-MONSOON ACCURACY:")
#print(df_accuracy_postmonsoon)

#calculate winter cpi
df_winter = df_seasonal[df_seasonal['season'] == 'winter']
df_accuracy_winter = calculate_accuracy(df_winter)
df_accuracy_winter['season'] = 'winter'
#print("WINTER ACCURACY:")
#print(df_accuracy_winter)

#concatenate all cpi
df_accuracy_final = pd.concat([df_accuracy_seasonal, df_accuracy_monsoon, df_accuracy_premonsoon, df_accuracy_postmonsoon, df_accuracy_winter])
df_accuracy_final

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n,season
0,ERA5L,0.747147,2.0,0.992208,0.483357,31.905380,1.243527,21,all
1,HAR,0.919864,3.0,0.971429,0.205839,12.586718,0.980854,21,all
2,CHIRPS,1.074379,5.0,0.968831,0.199289,-10.549457,0.914993,21,all
3,MSWEP,3.277778,6.0,0.985714,1.473589,-89.404981,0.095383,21,all
4,PERSIANN,0.981853,4.0,0.987013,0.498890,-24.930123,0.677887,21,all
5,kriged,0.740205,1.0,0.992208,0.480380,31.824891,1.240353,21,all
0,ERA5L,1.123753,1.0,0.600000,0.290164,26.755673,1.153398,5,monsoon
1,HAR,1.154319,3.0,0.600000,0.088614,2.343420,0.440274,5,monsoon
2,CHIRPS,1.221332,4.0,0.400000,0.118786,-8.899657,0.766035,5,monsoon
3,MSWEP,3.000000,6.0,0.900000,0.907929,-90.234596,0.113580,5,monsoon


**Accuracy and CPI Calculation - Annual Scale**

In [10]:
df_merged2 = df_merged.copy()
df_merged2['date'] = pd.to_datetime(df_merged2['date'])
df_merged2['year'] = df_merged2['date'].dt.year
df_annual = df_merged2.groupby(by = ['year'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec', 'kriged_prec']].sum().reset_index()
df_accuracy_annual = calculate_accuracy(df_annual)
df_accuracy_annual

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.976201,3.0,0.7,0.328324,31.905380,0.950469,5
1,HAR,0.746994,1.0,0.8,0.135351,12.586718,0.598642,5
2,CHIRPS,1.377022,5.0,0.3,0.123790,-10.549457,0.636196,5
3,MSWEP,3.000000,6.0,1.0,0.898912,-89.404981,0.098457,5
4,PERSIANN,1.297991,4.0,0.6,0.256460,-24.930123,0.639637,5
5,kriged,0.961456,2.0,0.7,0.327710,31.824891,0.961632,5


**Accuracy and CPI Calculation - Monthly Scale**

In [11]:
df_merged3 = df_merged.copy()
df_merged3['date'] = pd.to_datetime(df_merged3['date'])
df_merged3['year'] = df_merged3['date'].dt.year
df_merged3['month'] = df_merged3['date'].dt.month
df_monthly = df_merged3.groupby(by = ['year','month'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec', 'kriged_prec']].sum().reset_index()
df_accuracy_monthly = calculate_accuracy(df_monthly)
df_accuracy_monthly

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.844259,4.0,0.987441,0.478410,31.905380,1.253655,60
1,HAR,0.322435,1.0,0.982328,0.281507,12.586718,0.984599,60
2,CHIRPS,0.561513,2.0,0.976549,0.252682,-10.549457,0.899541,60
3,MSWEP,4.000000,6.0,0.960822,1.328719,-89.404981,0.093702,60
4,PERSIANN,0.784180,3.0,0.990275,0.497015,-24.930123,0.650740,60
5,kriged,0.851552,5.0,0.987108,0.477440,31.824891,1.251779,60


**Calculation of Categorical Metrices for Dataset Evaluation**

In [12]:
import xarray as xr
import operator
import scores.categorical
df_merged

,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,kriged_prec,prec_insitu
0,2001-01-01,102.0,7.8,4.7,16.0,18.7,3.8,16.6,0.0
1,2001-01-01,103.0,8.3,3.9,15.4,5.7,3.9,15.3,0.0
2,2001-01-01,104.0,9.1,3.9,16.2,18.3,4.1,15.3,0.0
3,2001-01-01,105.0,9.9,0.0,3.2,1.5,1.6,3.2,0.0
4,2001-01-01,107.0,5.9,2.8,11.8,11.6,3.7,12.0,7.0
...,...,...,...,...,...,...,...,...,...
121380,2005-12-31,1407.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121381,2005-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121382,2005-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
121383,2005-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
#conversion to xarray
df_merged4 = df_merged.copy()

def calculate_categorical_metrics(dataframe_prec):
    df_categorical_metrics = pd.DataFrame()

    for dataset in datasets:
        dataset_categorical_metrics = {'dataset': dataset}

        obs_insitu = df_merged4.set_index(['stn_id', 'date'])['prec_insitu'].to_xarray()
        pred_dataset = df_merged4.set_index(['stn_id', 'date'])[f'{dataset}_prec'].to_xarray()

        #define event -  wet day is when precipitation is greater than 0.1
        event_operator = scores.categorical.ThresholdEventOperator(
            default_event_threshold=0.1, 
            default_op_fn=operator.ge)

        #conversion to binary
        dataset_binary, insitu_binary = event_operator.make_event_tables(pred_dataset, obs_insitu)
            
        #make contingency manager
        cm_dataset = event_operator.make_contingency_manager(dataset_binary, insitu_binary)
        
        #calculate metrics
        pod_dataset = cm_dataset.probability_of_detection()
        dataset_categorical_metrics['pod'] = pod_dataset.values
        far_dataset = cm_dataset.false_alarm_rate()
        dataset_categorical_metrics['far'] = far_dataset.values
        fb_dataset = cm_dataset.frequency_bias() 
        dataset_categorical_metrics['fb'] = fb_dataset.values
        csi_dataset = cm_dataset.critical_success_index()
        dataset_categorical_metrics['csi'] = csi_dataset.values
        
        # Wrap in a list to create a single-row DataFrame
        dataset_categorical_metrics = pd.DataFrame([dataset_categorical_metrics])
        df_categorical_metrics = pd.concat([df_categorical_metrics, dataset_categorical_metrics], ignore_index=True)

    return df_categorical_metrics

df_categorical_metrics = calculate_categorical_metrics(df_merged4)
numeric_cols = ['pod', 'far', 'fb', 'csi']
df_categorical_metrics[numeric_cols] = df_categorical_metrics[numeric_cols].astype(float).round(2)
df_categorical_metrics

,dataset,pod,far,fb,csi
0,ERA5L,0.98,0.68,2.69,0.36
1,HAR,0.85,0.35,1.74,0.45
2,CHIRPS,0.35,0.13,0.67,0.26
3,MSWEP,0.67,0.26,1.33,0.40
4,PERSIANN,0.78,0.39,1.77,0.39
5,kriged,0.98,0.69,2.71,0.36


In [20]:
# ══════════════════════════════════════════════════════════════════
# PRODUCT A FEASIBILITY DIAGNOSTIC — CELLS 1–4  (revised)
#
# Prerequisites in memory:
#   df_merged       — daily merged grid + insitu
#   df_insitu       — raw insitu with elevation/lat/long
#   seasons         — dict of season→month lists
#   calculate_accuracy — function defined above
#   df_accuracy_daily  — already computed above
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr, spearmanr

QUICK_RF = dict(n_estimators=200, max_depth=8,
                min_samples_leaf=5, n_jobs=-1, random_state=42)

def eval_model(y_true, y_pred, label):
    """Evaluate predictions; return dict and print one-liner."""
    r, _   = spearmanr(y_pred, y_true)
    nrmse  = np.sqrt(np.mean((y_pred - y_true)**2)) / (np.mean(y_true) + 1e-9)
    pbias  = 100 * np.sum(y_pred - y_true) / (np.sum(y_true) + 1e-9)
    r2     = r2_score(y_true, y_pred)
    print(f"  {label:<40} r={r:.3f}  nrmse={nrmse:.3f}  "
          f"pbias={pbias:+.1f}%  R²={r2:.3f}")
    return dict(label=label, spearman_r=round(r,3), nrmse=round(nrmse,3),
                pbias=round(pbias,1), r2=round(r2,3))

# ── shared station split (reused in Cells 3 & 4) ─────────────────
_df_split  = df_merged.dropna().copy()
_all_stns  = _df_split['stn_id'].unique()
_rng       = np.random.default_rng(42)
TRAIN_STNS = _rng.choice(_all_stns, size=int(0.7 * len(_all_stns)), replace=False)
TEST_STNS  = np.setdiff1d(_all_stns, TRAIN_STNS)

ALL_PROD_COLS = ['ERA5L_prec', 'CHIRPS_prec', 'HAR_prec',
                 'MSWEP_prec', 'PERSIANN_prec', 'kriged_prec']
AVAIL_COLS = [c for c in ALL_PROD_COLS if c in _df_split.columns]

In [21]:
# ── CELL 1: ERA5L SYSTEMATIC BIAS STRUCTURE ──────────────────────
# Question: does ERA5L bias have exploitable spatial structure?
# If R²(bias ~ lat/lon/elev) is low, no spatial correction can help.
# Secondary: does kriging change that structure at all?

print("═" * 65)
print("CELL 1: ERA5L SYSTEMATIC BIAS STRUCTURE")
print("═" * 65)

df_c1 = df_merged.copy()
df_c1['date']   = pd.to_datetime(df_c1['date'])
df_c1['month']  = df_c1['date'].dt.month
df_c1['season'] = df_c1['month'].apply(
    lambda x: next(k for k, v in seasons.items() if x in v))
df_c1['era5l_bias']  = df_c1['ERA5L_prec']  - df_c1['prec_insitu']
df_c1['kriged_bias'] = df_c1['kriged_prec'] - df_c1['prec_insitu']

# Per-station mean bias
stn_bias = df_c1.groupby('stn_id').agg(
    era5l_mean_bias  = ('era5l_bias',  'mean'),
    kriged_mean_bias = ('kriged_bias', 'mean'),
    n_days           = ('era5l_bias',  'count')
).reset_index()

stn_meta = (df_insitu[['index_no','lat','long','elevation']]
            .drop_duplicates()
            .rename(columns={'index_no':'stn_id'}))
stn_bias = stn_bias.merge(stn_meta, on='stn_id', how='left')
X_spatial = stn_bias[['lat','long','elevation']].fillna(0).values

# (a) Spatial structure of bias
for col, label in [('era5l_mean_bias','Raw ERA5L'),
                   ('kriged_mean_bias','Kriged ERA5L')]:
    y  = stn_bias[col].values
    lr = LinearRegression().fit(X_spatial, y)
    r2 = r2_score(y, lr.predict(X_spatial))
    print(f"\n  [{label}]")
    print(f"    Mean bias            : {y.mean():+.3f} mm/day")
    print(f"    Std of station biases: {y.std():.3f} mm/day")
    print(f"    Range                : {y.min():+.3f} → {y.max():+.3f} mm/day")
    print(f"    R²(bias ~ lat+lon+elev): {r2:.3f}", end="  ")
    if r2 > 0.20:
        print("→ SYSTEMATIC spatial bias — RF correction is motivated")
    elif r2 > 0.08:
        print("→ MODERATE spatial structure")
    else:
        print("→ WEAK spatial structure — bias is largely random")

# (b) Does kriging shift the bias pattern?
shift = (stn_bias['kriged_mean_bias'] - stn_bias['era5l_mean_bias']).abs().mean()
print(f"\n  Mean absolute bias shift (kriged − raw): {shift:.4f} mm/day")
print("  → Kriging does NOT change the bias pattern" if shift < 0.05
      else "  → Kriging shifts biases — spatial interpolation alters local estimates")

# (c) Seasonal bias
print(f"\n  ERA5L mean bias by season:")
for s in ['monsoon','pre_monsoon','post_monsoon','winter']:
    sub = df_c1[df_c1['season'] == s]
    print(f"    {s:<15}: ERA5L {sub['era5l_bias'].mean():+.3f}  "
          f"Kriged {sub['kriged_bias'].mean():+.3f} mm/day  (n={len(sub):,})")

# (d) Elevation quartile bias
stn_bias['elev_bin'] = pd.qcut(stn_bias['elevation'], q=4,
                                labels=['Low','Mid-low','Mid-high','High'])
print(f"\n  ERA5L mean bias by elevation quartile:")
print(stn_bias.groupby('elev_bin', observed=True)[
    ['era5l_mean_bias','kriged_mean_bias']].mean().round(3).to_string())

═════════════════════════════════════════════════════════════════
CELL 1: ERA5L SYSTEMATIC BIAS STRUCTURE
═════════════════════════════════════════════════════════════════

  [Raw ERA5L]
    Mean bias            : +1.355 mm/day
    Std of station biases: 1.804 mm/day
    Range                : -1.892 → +7.263 mm/day
    R²(bias ~ lat+lon+elev): 0.242  → SYSTEMATIC spatial bias — RF correction is motivated

  [Kriged ERA5L]
    Mean bias            : +1.351 mm/day
    Std of station biases: 1.749 mm/day
    Range                : -2.050 → +6.638 mm/day
    R²(bias ~ lat+lon+elev): 0.256  → SYSTEMATIC spatial bias — RF correction is motivated

  Mean absolute bias shift (kriged − raw): 0.1550 mm/day
  → Kriging shifts biases — spatial interpolation alters local estimates

  ERA5L mean bias by season:
    monsoon        : ERA5L +2.595  Kriged +2.564 mm/day  (n=40,731)
    pre_monsoon    : ERA5L +0.779  Kriged +0.815 mm/day  (n=30,561)
    post_monsoon   : ERA5L +0.830  Kriged +0.814 mm/da

In [22]:
# ── CELL 2: INTER-PRODUCT UNIQUE INFORMATION BEYOND ERA5L ────────
# Partial correlation of each product with insitu after removing
# variance already explained by ERA5L.
# This answers: does each product add *anything* ERA5L doesn't have?
# Kriged ERA5L is included explicitly — near-zero partial r confirms
# it is a spatial smoother, not an information source.

print("═" * 65)
print("CELL 2: INTER-PRODUCT UNIQUE INFORMATION BEYOND ERA5L")
print("═" * 65)

df_c2    = df_merged.dropna().copy()
X_e      = df_c2['ERA5L_prec'].values.reshape(-1, 1)
y_i      = df_c2['prec_insitu'].values
lr_base  = LinearRegression().fit(X_e, y_i)
insitu_resid = y_i - lr_base.predict(X_e)
r2_era5l = r2_score(y_i, lr_base.predict(X_e))
raw_r_era5l = pearsonr(df_c2['ERA5L_prec'].values, y_i)[0]

print(f"\n  ERA5L alone  →  Pearson r={raw_r_era5l:.3f}   R²={r2_era5l:.3f}")
print(f"\n  {'Product':<14} {'Raw r':>8} {'Partial r':>10} {'Unique info?':>14}")
print("  " + "-" * 50)

partial_results = {}
for prod in ['CHIRPS','HAR','MSWEP','PERSIANN','kriged']:
    col = f'{prod}_prec'
    if col not in df_c2.columns:
        print(f"  {prod:<14}  — column missing, skipped")
        continue
    x_prod    = df_c2[col].values
    raw_r     = pearsonr(x_prod, y_i)[0]
    lr_prod   = LinearRegression().fit(X_e, x_prod)
    prod_resid = x_prod - lr_prod.predict(X_e)
    part_r, part_p = pearsonr(prod_resid, insitu_resid)
    unique = "YES" if (abs(part_r) > 0.05 and part_p < 0.01) else "no"
    partial_results[prod] = dict(raw_r=raw_r, partial_r=part_r)
    print(f"  {prod:<14} {raw_r:>8.3f} {part_r:>10.3f} {unique:>14}")

if 'kriged' in partial_results:
    kr = partial_results['kriged']['partial_r']
    print(f"\n  Kriged partial r = {kr:.4f}  →  "
          + ("spatial smoother only — no new information" if abs(kr) < 0.05
             else "adds unique information beyond raw ERA5L"))

# Pairwise correlation matrix
avail_for_corr = [c for c in ALL_PROD_COLS if c in df_c2.columns]
corr = df_c2[avail_for_corr].corr()
corr.columns = corr.index = [c.replace('_prec','') for c in avail_for_corr]
print(f"\n  Pairwise Pearson r (collinearity check):")
print(corr.round(3).to_string())
n_high = int(((corr.values > 0.85).sum() - len(avail_for_corr)) / 2)
print(f"\n  Product pairs with r > 0.85: {n_high}  "
      + ("→ HIGH collinearity" if n_high >= 4 else "→ Sufficient independence for merge"))

═════════════════════════════════════════════════════════════════
CELL 2: INTER-PRODUCT UNIQUE INFORMATION BEYOND ERA5L
═════════════════════════════════════════════════════════════════

  ERA5L alone  →  Pearson r=0.404   R²=0.163

  Product           Raw r  Partial r   Unique info?
  --------------------------------------------------
  CHIRPS            0.284      0.172            YES
  HAR               0.336      0.145            YES
  MSWEP             0.200      0.082            YES
  PERSIANN          0.308      0.195            YES
  kriged            0.405      0.026             no

  Kriged partial r = 0.0259  →  spatial smoother only — no new information

  Pairwise Pearson r (collinearity check):
          ERA5L  CHIRPS    HAR  MSWEP  PERSIANN  kriged
ERA5L     1.000   0.335  0.557  0.319     0.348   0.996
CHIRPS    0.335   1.000  0.300  0.352     0.668   0.336
HAR       0.557   0.300  1.000  0.239     0.342   0.559
MSWEP     0.319   0.352  0.239  1.000     0.403   0.318
PE

In [23]:
# ── CELL 3: MERGE RF vs ERA5L BASELINE ───────────────────────────
# Five models:
#   M0 — ERA5L direct (no model)           ← true baseline
#   M1 — RF all products                   ← full pipeline ceiling
#   M2 — RF ERA5L + best partial-r product ← lean complement
#   M3 — RF ERA5L + kriged only            ← isolates kriging as predictor
#   M4 — RF ERA5L only                     ← isolates bias-correction component
#
# CRITICAL COMPARISONS (printed explicitly):
#   M0 → M4  : gain from non-linear bias correction of ERA5L alone
#   M4 → M1  : gain from adding satellite products (true merge test)
#   M3 → M4  : marginal gain from kriged ERA5L as predictor

print("═" * 65)
print("CELL 3: MERGE RF vs ERA5L BASELINE")
print("═" * 65)
print("  Station-held-out split (70/30 by station, not by row).\n")

df_c3 = df_merged.dropna().copy()
tr    = df_c3[df_c3['stn_id'].isin(TRAIN_STNS)]
te    = df_c3[df_c3['stn_id'].isin(TEST_STNS)]
y_tr, y_te = tr['prec_insitu'].values, te['prec_insitu'].values

print(f"  Train stations: {len(TRAIN_STNS)}  ({len(tr):,} rows)")
print(f"  Test  stations: {len(TEST_STNS)}  ({len(te):,} rows)\n")

res = {}  # keyed by model name for clean retrieval

# M0 — direct ERA5L, no model
res['M0'] = eval_model(y_te, te['ERA5L_prec'].values, 'M0: ERA5L direct (no model)')

# M1 — RF all available products
rf1 = RandomForestRegressor(**QUICK_RF).fit(tr[AVAIL_COLS].values, y_tr)
res['M1'] = eval_model(y_te, rf1.predict(te[AVAIL_COLS].values),
                       'M1: RF all products')

# M2 — RF ERA5L + best partial-r complement (excluding kriged)
non_kriged = {p: v for p, v in partial_results.items() if p != 'kriged'}
if non_kriged:
    best_prod = max(non_kriged, key=lambda p: abs(non_kriged[p]['partial_r']))
    best_col  = f'{best_prod}_prec'
    if best_col in df_c3.columns:
        rf2 = RandomForestRegressor(**QUICK_RF).fit(
            tr[['ERA5L_prec', best_col]].values, y_tr)
        res['M2'] = eval_model(y_te, rf2.predict(te[['ERA5L_prec', best_col]].values),
                               f'M2: RF ERA5L + {best_prod} (best complement)')

# M3 — RF ERA5L + kriged (isolates kriging contribution)
if 'kriged_prec' in df_c3.columns:
    rf3 = RandomForestRegressor(**QUICK_RF).fit(
        tr[['ERA5L_prec','kriged_prec']].values, y_tr)
    res['M3'] = eval_model(y_te, rf3.predict(te[['ERA5L_prec','kriged_prec']].values),
                           'M3: RF ERA5L + kriged only')

# M4 — RF ERA5L alone (isolates bias-correction)
rf4 = RandomForestRegressor(**QUICK_RF).fit(tr[['ERA5L_prec']].values, y_tr)
res['M4'] = eval_model(y_te, rf4.predict(te[['ERA5L_prec']].values),
                       'M4: RF ERA5L only (bias-correction)')

# ── Gain decomposition (correct comparisons) ─────────────────────
gain_bc    = res['M4']['r2'] - res['M0']['r2']   # M0→M4: bias-correction
gain_merge = res['M1']['r2'] - res['M4']['r2']   # M4→M1: true merge test
gain_total = res['M1']['r2'] - res['M0']['r2']   # M0→M1: total (do not use for verdict)
gain_krige = (res['M3']['r2'] - res['M4']['r2']
              if 'M3' in res else None)            # M3→M4: kriging as predictor

print(f"\n  ── Gain decomposition ──────────────────────────────────")
print(f"  Bias-correction only  (M0 → M4) : {gain_bc:+.3f}  ← dominant?")
print(f"  Multi-source merging  (M4 → M1) : {gain_merge:+.3f}  ← true merge test")
print(f"  Total pipeline        (M0 → M1) : {gain_total:+.3f}  (= sum of above)")
if gain_krige is not None:
    print(f"  Kriging as predictor  (M3 → M4) : {gain_krige:+.3f}  ← add kriged_prec?")

# ── Verdicts ─────────────────────────────────────────────────────
print(f"\n  ── Verdicts ────────────────────────────────────────────")

# Bias-correction verdict
if gain_bc > 0.30:
    print(f"  [BIAS-CORRECTION] STRONG ({gain_bc:+.3f}) — "
          "RF substantially corrects ERA5L systematic error")
elif gain_bc > 0.10:
    print(f"  [BIAS-CORRECTION] MODERATE ({gain_bc:+.3f})")
else:
    print(f"  [BIAS-CORRECTION] WEAK ({gain_bc:+.3f}) — "
          "ERA5L is already near the daily-scale ceiling")

# Merge verdict — based on M4→M1, NOT M0→M1
if gain_merge > 0.05:
    print(f"  [MERGING] MEANINGFUL ({gain_merge:+.3f}) — "
          "satellite products contribute beyond ERA5L bias-correction")
elif gain_merge > 0.02:
    print(f"  [MERGING] MARGINAL ({gain_merge:+.3f}) — "
          "modest added skill; justify by pbias or season-specific gains")
else:
    print(f"  [MERGING] NEGLIGIBLE ({gain_merge:+.3f}) — "
          "Product A is a bias-correction product, not a multi-source merge.\n"
          "  Revise framing accordingly.")

# Kriging verdict — based on M3→M4
if gain_krige is not None:
    if abs(gain_krige) < 0.005:
        print(f"  [KRIGING] NEGLIGIBLE ({gain_krige:+.3f}) — "
              "drop kriged_prec as RF feature; it replicates ERA5L_prec")
    elif gain_krige > 0:
        print(f"  [KRIGING] MARGINAL POSITIVE ({gain_krige:+.3f}) — "
              "keep if pipeline cost is low, but do not claim as novel predictor")
    else:
        print(f"  [KRIGING] NEGATIVE ({gain_krige:+.3f}) — "
              "kriged_prec actively hurts; remove")

# pbias comparison table
print(f"\n  ── pbias summary (wet-bias reduction) ──────────────────")
print(f"  {'Model':<42} {'R²':>6}  {'pbias':>8}  {'Δ R² vs M0':>11}")
print("  " + "─" * 72)
for k, v in res.items():
    delta = v['r2'] - res['M0']['r2']
    print(f"  {v['label']:<42} {v['r2']:>6.3f}  {v['pbias']:>+7.1f}%  {delta:>+11.3f}")

═════════════════════════════════════════════════════════════════
CELL 3: MERGE RF vs ERA5L BASELINE
═════════════════════════════════════════════════════════════════
  Station-held-out split (70/30 by station, not by row).

  Train stations: 47  (82,261 rows)
  Test  stations: 21  (37,338 rows)

  M0: ERA5L direct (no model)              r=0.562  nrmse=3.441  pbias=+44.4%  R²=-0.409
  M1: RF all products                      r=0.575  nrmse=2.554  pbias=+14.8%  R²=0.224
  M2: RF ERA5L + PERSIANN (best complement) r=0.566  nrmse=2.578  pbias=+12.4%  R²=0.209
  M3: RF ERA5L + kriged only               r=0.562  nrmse=2.611  pbias=+11.7%  R²=0.189
  M4: RF ERA5L only (bias-correction)      r=0.561  nrmse=2.620  pbias=+11.9%  R²=0.183

  ── Gain decomposition ──────────────────────────────────
  Bias-correction only  (M0 → M4) : +0.592  ← dominant?
  Multi-source merging  (M4 → M1) : +0.041  ← true merge test
  Total pipeline        (M0 → M1) : +0.633  (= sum of above)
  Kriging as predicto

In [24]:
# ── CELL 4: SEASON-STRATIFIED MERGE TEST ─────────────────────────
# Runs M0, M4, M1 by season on the same station-held-out test set.
# M4 column is the key addition vs original — it separates
# bias-correction gain from merge gain within each season.
#
# Key question: is the M4→M1 merge gain concentrated in one season,
# or spread across all? If concentrated (e.g. pre-monsoon), that
# season is where the satellite products genuinely add information.

print("═" * 65)
print("CELL 4: SEASON-STRATIFIED PERFORMANCE (M0 / M4 / M1)")
print("═" * 65)

df_c4          = df_merged.dropna().copy()
df_c4['date']  = pd.to_datetime(df_c4['date'])
df_c4['month'] = df_c4['date'].dt.month
df_c4['season']= df_c4['month'].apply(
    lambda x: next(k for k, v in seasons.items() if x in v))

# Reuse trained models from Cell 3 (rf1=M1, rf4=M4)
# If Cell 3 was not run first, raise a clear error
for name, model in [('rf1 (M1)', rf1), ('rf4 (M4)', rf4)]:
    assert model is not None, f"{name} not found — run Cell 3 first"

te_c4 = df_c4[df_c4['stn_id'].isin(TEST_STNS)]

print(f"\n  {'Season':<15} {'n':>6} "
      f"{'M0 R²':>8} {'M4 R²':>8} {'M1 R²':>8} "
      f"{'ΔBC':>7} {'ΔMrg':>7} "
      f"{'M0 pb%':>8} {'M4 pb%':>8} {'M1 pb%':>8}")
print("  " + "─" * 90)

season_rows = []
for s in ['monsoon','pre_monsoon','post_monsoon','winter']:
    sub = te_c4[te_c4['season'] == s]
    if len(sub) < 50:
        print(f"  {s:<15}  — too few rows ({len(sub)}), skipped")
        continue

    y_true  = sub['prec_insitu'].values
    y_m0    = sub['ERA5L_prec'].values
    y_m4    = rf4.predict(sub[['ERA5L_prec']].values)
    y_m1    = rf1.predict(sub[AVAIL_COLS].values)

    def _r2(yt, yp):  return r2_score(yt, yp)
    def _pb(yt, yp):  return 100*np.sum(yp-yt)/(np.sum(yt)+1e-9)

    r2_m0, r2_m4, r2_m1 = _r2(y_true,y_m0), _r2(y_true,y_m4), _r2(y_true,y_m1)
    pb_m0, pb_m4, pb_m1 = _pb(y_true,y_m0), _pb(y_true,y_m4), _pb(y_true,y_m1)
    d_bc  = r2_m4 - r2_m0   # bias-correction gain
    d_mrg = r2_m1 - r2_m4   # merging gain

    print(f"  {s:<15} {len(sub):>6,} "
          f"{r2_m0:>8.3f} {r2_m4:>8.3f} {r2_m1:>8.3f} "
          f"{d_bc:>+7.3f} {d_mrg:>+7.3f} "
          f"{pb_m0:>+8.1f}% {pb_m4:>+8.1f}% {pb_m1:>+8.1f}%")
    season_rows.append(dict(season=s, r2_m0=r2_m0, r2_m4=r2_m4, r2_m1=r2_m1,
                            delta_bc=d_bc, delta_merge=d_mrg,
                            pb_m0=pb_m0, pb_m4=pb_m4, pb_m1=pb_m1))

# Season-level summary verdict
if season_rows:
    df_seas = pd.DataFrame(season_rows)
    best_bc  = df_seas.loc[df_seas['delta_bc'].idxmax(),  'season']
    best_mrg = df_seas.loc[df_seas['delta_merge'].idxmax(),'season']
    print(f"\n  Bias-correction gain largest in : {best_bc}")
    print(f"  Merge gain largest in           : {best_mrg}")
    any_neg_merge = df_seas[df_seas['delta_merge'] < -0.01]['season'].tolist()
    if any_neg_merge:
        print(f"  WARNING — satellite products HURT R² in: {any_neg_merge}")
        print(f"  Consider season-specific feature selection for those seasons.")

# ── OVERALL FEASIBILITY VERDICT ───────────────────────────────────
print(f"\n{'═' * 65}")
print("PRODUCT A FEASIBILITY VERDICT (based on Cells 1–4)")
print(f"{'═' * 65}")

c1_r2_bias = None  # populated if Cell 1 was run with r2 stored
n_unique   = sum(1 for p, v in partial_results.items()
                 if p != 'kriged' and abs(v['partial_r']) > 0.05)

print(f"\n  Cell 2 — products with unique info beyond ERA5L : {n_unique}/4")
print(f"  Cell 3 — bias-correction gain (M0→M4)          : {gain_bc:+.3f}")
print(f"  Cell 3 — merging gain         (M4→M1)          : {gain_merge:+.3f}")
if gain_krige is not None:
    print(f"  Cell 3 — kriging contribution (M3→M4)          : {gain_krige:+.3f}")

print()
if gain_bc > 0.10 and gain_merge > 0.05 and n_unique >= 2:
    print("  ✓ PROCEED as multi-source merge product")
    print("    Bias-correction is strong AND satellite products add genuine skill.")
elif gain_bc > 0.10 and gain_merge <= 0.05:
    print("  ⚠ REFRAME as ERA5L bias-correction product")
    print("    Dominant gain is non-linear bias correction of ERA5L alone.")
    print("    Satellite products may still be retained as secondary features,")
    print("    but the central claim must be bias correction, not merging.")
    print("    Consider: does M2 (ERA5L + best complement) justify complexity of M1?")
elif gain_bc <= 0.10:
    print("  ✗ ERA5L is near the daily-scale information ceiling")
    print("    Neither bias-correction nor merging adds meaningful R².")
    print("    Proceed directly to Product B (monthly downscaled).")

═════════════════════════════════════════════════════════════════
CELL 4: SEASON-STRATIFIED PERFORMANCE (M0 / M4 / M1)
═════════════════════════════════════════════════════════════════

  Season               n    M0 R²    M4 R²    M1 R²     ΔBC    ΔMrg   M0 pb%   M4 pb%   M1 pb%
  ──────────────────────────────────────────────────────────────────────────────────────────
  monsoon         12,560   -0.592    0.099    0.154  +0.690  +0.055    +42.0%     +0.9%     +6.2%
  pre_monsoon      9,396   -0.273    0.046    0.075  +0.319  +0.028    +31.8%    +26.0%    +27.2%
  post_monsoon     6,135   -1.067    0.110    0.063  +1.177  -0.047    +92.6%    +74.4%    +67.3%
  winter           9,247   -0.177    0.039    0.053  +0.217  +0.014    +79.2%    +89.5%    +69.4%

  Bias-correction gain largest in : post_monsoon
  Merge gain largest in           : monsoon
  WARNING — satellite products HURT R² in: ['post_monsoon']
  Consider season-specific feature selection for those seasons.

═══════════════